In [1]:
import os
import zipfile
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 경로 설정
ZIP_SOURCE_DIR = '/content/drive/MyDrive/tuk_YOLO_data' # zip 파일들이 있는 폴더 경로
DATASET_DEST_DIR = '/content/dataset'

In [3]:
# 압축 풀기
os.makedirs(DATASET_DEST_DIR, exist_ok=True)
for root, dirs, files in os.walk(ZIP_SOURCE_DIR):
    for file in files:
        if file.endswith('.zip'):
            zip_path = os.path.join(root, file)
            print(f"압축 해제 중: {file}")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(DATASET_DEST_DIR)

print("\n모든 데이터셋 통합 완료")

압축 해제 중: corridor_second_floor.zip
압축 해제 중: corridor_third_floor2.zip
압축 해제 중: corridor_third_floor.zip
압축 해제 중: corridor_fourth_floor.zip
압축 해제 중: corridor_seventh_floor.zip
압축 해제 중: corridor_fifth_floor.zip
압축 해제 중: corridor_sixth_floor.zip

모든 데이터셋 통합 완료


In [7]:
# 라이브러리 설치
!pip install ultralytics

import os
import shutil
import glob
from ultralytics import YOLO

In [8]:
# 경로 설정 및 필수 폴더 생성
DATASET_ROOT = '/content/dataset'
TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train/images')
TRAIN_LBL_DIR = os.path.join(DATASET_ROOT, 'train/labels')

os.makedirs(TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(TRAIN_LBL_DIR, exist_ok=True)

# 흩어져 있는 이미지와 라벨을 새로 만든 폴더로 강제 이동
all_images = glob.glob(f"{DATASET_ROOT}/**/*.jpg", recursive=True) + \
             glob.glob(f"{DATASET_ROOT}/**/*.png", recursive=True)
all_labels = glob.glob(f"{DATASET_ROOT}/**/*.txt", recursive=True)

print(f"찾은 이미지 개수: {len(all_images)}개")
print(f"찾은 라벨 개수: {len(all_labels)}개")

for img in all_images:
    if 'train/images' not in img: # 이미 옮겨진 건 제외
        shutil.move(img, os.path.join(TRAIN_IMG_DIR, os.path.basename(img)))
for lbl in all_labels:
    if 'train/labels' not in lbl: # 이미 옮겨진 건 제외
        shutil.move(lbl, os.path.join(TRAIN_LBL_DIR, os.path.basename(lbl)))

# 3. data.yaml 새로 생성 (가장 안전한 방식)
yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
yaml_content = f"""
train: {TRAIN_IMG_DIR}
val: {TRAIN_IMG_DIR}  # 검증 데이터가 없으면 학습용을 같이 사용

nc: 14
names: ['elevator', 'board', 'locker', 'vending_machine', 'trash_bin', 'door', 'book_shelf', 'signboard', 'self_service_cafe', 'water_dispenser', 'white_locker', 'paper_box', 'column', 'bench']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print("데이터 정리 및 data.yaml 생성 완료")

# 4. YOLOv8n 학습 시작
model = YOLO('yolov8n.pt')
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    name='hallway_project',
    device=0  # GPU 사용
)

찾은 이미지 개수: 3939개
찾은 라벨 개수: 3081개
데이터 정리 및 data.yaml 생성 완료
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hallway_project2, nbs=64, nms=False, opset=None, optimize=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fed3d1e42f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.0460